# TimesFM → NanoJev: признаки из эмбеддингов — в Jev-модель, с головой на выходе

**Задача та же:** по окну потребления мощности (P_RMS, мВт) умной розетки определить **паттерн поведения** (класс из имени файла `plug_dump_..._<LABEL>_<DURATION>.csv`).

**Идея «Франкенштейна» (как в `Jev_Laya_smartplug.ipynb`, но для NanoJev):**
1. **TimesFM 3.0** (заморожен) энкодит 90-точечное окно → патч-эмбеддинги → **маскированный mean-pool** → вектор **1280** — это и есть «признаки» окна.
2. Эти признаки попадают в **NanoJev** двумя путями:
   - **A. Zero-shot через state**: признаки сжимаем в текст (StandardScaler → PCA на 16 главных компонент + прежние статистики) и кладём в текстовый `state` модели. NanoJev не меняется — это «передаём признаки в NanoJev».
   - **B. Голова на выходе NanoJev**: замораживаем backbone (Qwen3-0.6B) чекпойнта, прогоняем candidate-пути того же choice-вопроса, берём скрытые векторы **последних позиций** (то, что в оригинале уходит в decision-голову `LayerNorm→Linear(hidden,1)`), **mean-pool по 4 кандидатам** каждого окна → вектор `H` — и обучаем **небольшую классификационную голову** (Linear + CE) поверх этого вектора.
3. **«Потолок»** — RF/LogReg прямо на 1280-мерных эмбеддингах (без Jev-механики): показывает, насколько признаки TimesFM отделимы вообще.

Всё сравнивается на **одном тесте** (стратифицированный сплит 30%, как в исходнике).

> ⚠️ Нужен GPU (T4+): TimesFM и NanoJev оба на CUDA. Нужен интернет для скачивания чекпойнтов (`google/timesfm-3.0-pytorch`, `C-Tianyu/NanoJev`).

## 0. Окружение и данные

In [ ]:
# 0.1 Базовые зависимости
#     transformers >= 4.46 обязателен: чекпойнт NanoJev использует Qwen2-токенизатор с
#     list-стилем extra_special_tokens (в старых версиях transformers он не загружается).
%pip install -q "transformers>=4.46" torch safetensors huggingface_hub numpy pandas scikit-learn matplotlib tqdm
print("базовые зависимости установлены (timesfm ставится отдельно в 2.1)")

In [ ]:
# 0.2 Импорты и общие настройки
import os, sys, re, json, warnings, subprocess
from collections import Counter

os.environ.setdefault("USE_TF", "0")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("Среда:", "Google Colab" if IN_COLAB else "Локальная (VS Code)")

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Устройство:", DEVICE, "| CUDA:", torch.cuda.is_available())
except ImportError:
    DEVICE = "cpu"
    print("Устройство: cpu (torch не найден)")

In [ ]:
# 0.3 Папка с данными
DATA_DIR = os.environ.get("SMARTPLUG_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DATA_DIR or "/content/drive/MyDrive/SmartPlug"
else:
    DATA_DIR = DATA_DIR or os.path.expanduser("~/SmartPlug")

print("Папка данных:", DATA_DIR)
if not os.path.isdir(DATA_DIR):
    raise SystemExit(
        "Папка с данными не найдена. В Colab положите датасет в MyDrive/SmartPlug; "
        "локально — укажите SMARTPLUG_DIR или создайте ~/SmartPlug."
    )

## 1. Загрузка данных (как в исходном ноутбуке)

In [ ]:
# 1.1 Параметры разбиения
CHUNK_LENGTH = 90
FILTER_OUT = ("IdleCharge", "MixedBrowsing", "Browsing", "VKAudio")
COLUMN = " P_RMS, mW"

filelist = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv"))
print("Найдено файлов:", len(filelist))
if not filelist:
    raise SystemExit("В папке данных нет .csv-файлов.")

In [ ]:
# 1.2 Парсер имени файла: plug_dump_YYYY_MM_DD_hh_mm_ss_<LABEL>_<DURATION>.csv
def get_label_duration(filename: str):
    duration = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\D+?_(\d+)\.csv", filename)
    label = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_(\D+?)_\d+\.csv", filename)
    if not duration or not label:
        return None, None
    return label[0], duration[0]

print("Пример имени:", filelist[0], "-> метка:", get_label_duration(filelist[0]))

In [ ]:
# 1.3 Чтение файлов и сбор окон длиной CHUNK_LENGTH
Labels, ndata, skipped = [], None, 0

for filename in tqdm(filelist, desc="Обработка файлов"):
    label, _ = get_label_duration(filename)
    if label is None:
        skipped += 1
        continue
    if any(tag in label for tag in FILTER_OUT):
        continue
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, filename), delimiter=";")
        col = next((c for c in df.columns if c.strip() == COLUMN.strip()), None)
        if col is None:
            skipped += 1
            continue
        series = df[col].to_numpy(dtype=np.float64)
    except Exception:
        skipped += 1
        continue

    num_chunks = len(series) // CHUNK_LENGTH
    for i in range(num_chunks):
        chunk = series[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]
        ndata = chunk[None, :] if ndata is None else np.vstack([ndata, chunk])
        Labels.append(label)

ndata = np.asarray(ndata, dtype=np.float32) if ndata is not None else np.empty((0, CHUNK_LENGTH))
print("Пропущено файлов:", skipped)
print("Всего фрагментов:", len(Labels), "| Форма:", ndata.shape)
print("Метки:")
for u, c in Counter(Labels).items():
    print(f"  {u}: {c}")
if len(Labels) < 20:
    raise SystemExit("Слишком мало фрагментов.")

In [ ]:
# 1.4 Кодирование меток и стратифицированный сплит train/test
le = LabelEncoder()
y_encoded = le.fit_transform(Labels)
classes = list(le.classes_)
print("Классов:", len(classes), "|", ", ".join(classes))

X_train, X_test, y_train, y_test = train_test_split(
    ndata, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
test_labels = le.inverse_transform(y_test)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. TimesFM 3.0 — признаки для каждого окна

Берём из `Jev_Laya_smartplug.ipynb` проверенный путь: z-нормализация ряда → патчи (32 точки) → один forward backbone → `transformer_output` `(B, n_patches, 1280)` → **маскированный mean-pool** (только реальные точки патчей) → вектор 1280 на окно.

In [ ]:
# 2.1 Установка и загрузка TimesFM 3.0 (backbone)
#     API тот же, что в исходном ноутбуке TimesFM3_DeepSeek.ipynb.
%pip install -q timesfm
from timesfm3 import TimesFM3Forecaster

forecaster = TimesFM3Forecaster.from_pretrained(
    "google/timesfm-3.0-pytorch",
    per_core_batch_size=4,
)
print(f"TimesFM загружен на устройстве: {forecaster.device}")

backbone_tf = forecaster.model
HIDDEN_TF = 1280
INPUT_PATCH = backbone_tf.input_patch_len   # 32
print("input_patch_len:", INPUT_PATCH, "| hidden_dim:", HIDDEN_TF)

In [ ]:
# 2.2 Патчинг + замороженный энкодер с z-норм и маскированным mean-pool.
import torch
import torch.nn as nn

def pad_to_patches(x: torch.Tensor):
    '''x: (B, L) -> values (B, 1, n_patches, p) + masks + patch_is_target.'''
    batch_size, seq_len = x.shape
    n_patches = (seq_len + INPUT_PATCH - 1) // INPUT_PATCH
    padded_len = n_patches * INPUT_PATCH
    pad = padded_len - seq_len
    x_padded = torch.nn.functional.pad(x, (0, pad)) if pad > 0 else x
    mask = torch.zeros(batch_size, 1, padded_len, dtype=torch.bool, device=x.device)
    mask[:, :, seq_len:] = True
    values = x_padded.view(batch_size, 1, n_patches, INPUT_PATCH)
    masks = mask.view(batch_size, 1, n_patches, INPUT_PATCH)
    patch_is_target = torch.ones(batch_size, 1, n_patches, dtype=torch.bool, device=x.device)
    return {'values': values, 'masks': masks, 'patch_is_target': patch_is_target}

class TimesFMEncoder(nn.Module):
    '''Замороженный TimesFM: z-норм + патчи + masked mean-pool -> (B, 1280).'''
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)  # z-норм
        inputs = pad_to_patches(x)
        out = self.backbone.forward(inputs, return_aux_outputs=True)
        patches = out['__call__:transformer_output'].squeeze(1)   # (B, n_patches, 1280)
        real = (~inputs['masks'].squeeze(1)).float()              # (B, n_patches, P)
        w = real.sum(-1)
        w = w / w.sum(-1, keepdim=True).clamp(min=1e-9)
        return (patches * w.unsqueeze(-1)).sum(1)                 # masked mean-pool (B, 1280)

tf_enc = TimesFMEncoder(backbone_tf).to(DEVICE)
print("Encoder TimesFM-Lite готов. hidden:", HIDDEN_TF)

def embed_tf(model, X, device, batch=64):
    model.eval()
    outs = []
    for xb in DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch):
        outs.append(model(xb.to(device)).detach().cpu().numpy())
    return np.vstack(outs)

In [ ]:
# 2.2b Импорт DataLoader (для embed_tf) и извлечение эмбеддингов train/test.
from torch.utils.data import DataLoader

print("Извлекаем эмбеддинги TimesFM (может занять пару минут)...")
Etr = embed_tf(tf_enc, X_train, DEVICE)
Ete = embed_tf(tf_enc, X_test,  DEVICE)
print("E_train:", Etr.shape, "| E_test:", Ete.shape)

In [ ]:
# 2.3 «Потолок» признаков: RF/LogReg прямо на эмбеддингах (без Jev-механики).
#     Если потолок высокий, а NanoJev/голова ниже — значит, теряет транспортировка признаков,
#     а не сами признаки.
sc_emb = StandardScaler().fit(Etr)
Etr_s, Ete_s = sc_emb.transform(Etr), sc_emb.transform(Ete)

probe = {}
for name, clf in [
    ("LogisticRegression", LogisticRegression(max_iter=2000, C=1.0)),
    ("RandomForest",       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
]:
    clf.fit(Etr_s, y_train)
    p = clf.predict(Ete_s)
    probe[name] = {"acc": accuracy_score(y_test, p), "f1": f1_score(y_test, p, average="macro")}
    print(f"[Probe {name}] acc={probe[name]['acc']:.4f}  macro-F1={probe[name]['f1']:.4f}")

In [ ]:
# 2.4 PCA: 1280-мерные признаки -> 16 главных компонент (для текстового state NanoJev).
#     Калибруем ТОЛЬКО на train, тест только трансформируем — честно, без утечки.
N_COMP = 16
pca = PCA(n_components=N_COMP, random_state=42).fit(Etr_s)
Ctr = pca.transform(Etr_s)   # (n_train, 16)
Cte = pca.transform(Ete_s)   # (n_test,  16)
print("Компоненты:", Ctr.shape, Cte.shape)
print("Объяснённая дисперсия (суммарно): %.3f" % pca.explained_variance_ratio_.sum())
print("Первые 3 компоненты первого train-окна:", np.round(Ctr[0, :3], 3))

## 3. NanoJev zero-shot: статистики vs признаки TimesFM в текстовом state

Два варианта `state` для одного и того же choice-вопроса:
- **stats** — прежний формат (11 статистик + прорежённая кривая), как в `NanoJev_vs_RF_smartplug.ipynb`;
- **tfmpca** — те же статистики **+** «TimesFM embedding (standardized, top PCA): c1=…, c2=…, …, c16=…».

Это прямой ответ на «передать признаки TimesFM в NanoJev» без изменения весов модели.

In [ ]:
# 3.1 Клонируем репозиторий NanoJev (код + dataset; веса ниже с HF).
REPO_DIR = "/content/NanoJev" if IN_COLAB else os.path.expanduser("~/NanoJev")
if not os.path.isdir(os.path.join(REPO_DIR, "scripts")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TianyuCodings/NanoJev.git", REPO_DIR], check=True)
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")
sys.path.insert(0, SCRIPTS_DIR)
print("scripts:", SCRIPTS_DIR)

In [ ]:
# 3.2 Скачиваем чекпойнт unified-games-v1.
from huggingface_hub import snapshot_download

CKPT_DIR = os.environ.get("NANOJEV_CKPT", "").strip() or os.path.join(REPO_DIR, "checkpoints", "NanoJev-unified")
if not os.path.isdir(CKPT_DIR) or not os.path.isfile(os.path.join(CKPT_DIR, "best.safetensors")):
    print("Скачиваем C-Tianyu/NanoJev (revision=unified-games-v1)...")
    snapshot_download(
        repo_id="C-Tianyu/NanoJev",
        revision="unified-games-v1",
        local_dir=CKPT_DIR,
        allow_patterns=["best.safetensors", "config.json", "tokenizer/*", "backbone_config/*"],
    )
print("Чекпойнт:", CKPT_DIR)

In [ ]:
# 3.3 Загрузка DecisionPredictor (CUDA обязательна).
from predict_toy_decisions import DecisionPredictor

if DEVICE != "cuda":
    print("NanoJev требует CUDA. Подключите GPU (Colab: Runtime -> Change runtime type -> T4).")
    NANOJEV_READY = False
else:
    try:
        nj = DecisionPredictor(CKPT_DIR, precision="bf16")
        _reason = "bf16"
    except Exception as e1:
        print("Прямая загрузка не удалась, пробуем disable_native_triton=True:")
        print("  ", type(e1).__name__, str(e1)[:200])
        nj = DecisionPredictor(CKPT_DIR, precision="bf16", disable_native_triton=True)
        _reason = "bf16 + disable_native_triton"
    NANOJEV_READY = True
    print("DecisionPredictor готов:", _reason)
    print("base_model:", nj.run_config.get("model"), "| max_length:", nj.limit,
          "| set_head:", nj.run_config["set_head"])

In [ ]:
# 3.4 Текстовые state и choice-вопрос по фактическим классам.
CLASS_DESCRIPTIONS = {
    "IdleCharge":    "device is plugged in and idle or charging with low, stable power draw",
    "MixedBrowsing": "charging while doing light computer activity",
    "Browsing":      "active web browsing with moderate power draw",
    "VKAudio":       "audio or video streaming playback",
}
def class_desc(c: str) -> str:
    return CLASS_DESCRIPTIONS.get(c, f"home-appliance behaviour pattern {c}")

QUESTIONS_NJ = {"pattern": {
    "type": "choice",
    "instructions": ("Which home-appliance behaviour pattern does this power-consumption "
                     "window belong to? Judge from the statistics, the sampled curve and the "
                     "embedding components, then choose exactly one of the listed patterns."),
    "criteria": {c: class_desc(c) for c in classes},
}}
print("Кандидаты:", list(QUESTIONS_NJ["pattern"]["criteria"]))

def chunk_stats(chunk: np.ndarray) -> dict:
    '''Те же 11 статистик, что уходили в RF-бейзлайн.'''
    chunk = np.asarray(chunk, dtype=np.float64)
    return {
        "mean_mw": float(np.mean(chunk)), "std_mw": float(np.std(chunk)),
        "min_mw": float(np.min(chunk)), "max_mw": float(np.max(chunk)),
        "p25_mw": float(np.percentile(chunk, 25)), "p50_mw": float(np.percentile(chunk, 50)),
        "p75_mw": float(np.percentile(chunk, 75)), "range_mw": float(np.ptp(chunk)),
        "energy_g": float(np.sum(chunk ** 2) / 1e6),
        "mean_abs_diff": float(np.mean(np.abs(np.diff(chunk)))),
        "trend_mw_per_pt": float(np.polyfit(np.arange(len(chunk)), chunk, 1)[0]),
    }

def chunk_to_state(chunk: np.ndarray, id_: int, extra_line: str = None, n_points: int = 16) -> str:
    '''Числовой чанк -> текстовый state NanoJev (статистики + кривая + опц. доп.строка).'''
    s = chunk_stats(chunk)
    pts = np.round(chunk[::max(1, len(chunk) // n_points)][:n_points], 1)
    base = (
        f"Power-consumption window #{id_} from a smart plug, P_RMS in mW, 90 samples.\n"
        f"Statistics: mean={s['mean_mw']:.1f}, std={s['std_mw']:.1f}, min={s['min_mw']:.1f}, "
        f"max={s['max_mw']:.1f}, p25={s['p25_mw']:.1f}, median={s['p50_mw']:.1f}, "
        f"p75={s['p75_mw']:.1f}, range={s['range_mw']:.1f}, energy_x1e-6={s['energy_g']:.2f}, "
        f"mean_abs_step={s['mean_abs_diff']:.2f}, trend_mw_per_pt={s['trend_mw_per_pt']:.3f}.\n"
        f"Sampled curve mW: " + ", ".join(str(float(v)) for v in pts) + "."
    )
    if extra_line:
        base += "\n" + extra_line
    return base

def tfmpca_to_text(comp_row: np.ndarray) -> str:
    comps = ", ".join(f"c{i+1}={float(v):.3f}" for i, v in enumerate(comp_row))
    return "TimesFM embedding (standardized, top PCA components): " + comps

# Два варианта state для теста
n = len(X_test)
state_stats = [chunk_to_state(X_test[i], i) for i in range(n)]
state_tfmpca = [chunk_to_state(X_test[i], i, tfmpca_to_text(Cte[i])) for i in range(n)]
print("Версий state:", len(state_stats))
print("Пример tfmpca (первые 200 симв.):", state_tfmpca[0][:200], "...")

In [ ]:
# 3.4b Бейзлайн: Random Forest на тех же 11 статистиках (полная train/test выборка).
#     Те же признаки, что идут в "state" NanoJev, — честное сравнение на входе.
STAT_KEYS = ["mean_mw", "std_mw", "min_mw", "max_mw", "p25_mw", "p50_mw", "p75_mw",
             "range_mw", "energy_g", "mean_abs_diff", "trend_mw_per_pt"]

def stats_vector(chunk: np.ndarray) -> np.ndarray:
    s = chunk_stats(chunk)
    return np.array([s[k] for k in STAT_KEYS], dtype=np.float64)

X_feat = np.vstack([stats_vector(x) for x in tqdm(ndata, desc="Стат-признаки")])
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_feat, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
rf_scaler = StandardScaler().fit(Xf_tr)
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(rf_scaler.transform(Xf_tr), yf_tr)
rf_preds_full = rf_model.predict(rf_scaler.transform(Xf_te))
rf_acc = accuracy_score(yf_te, rf_preds_full)
rf_f1 = f1_score(yf_te, rf_preds_full, average="macro")
print(f"[RF stats] acc={rf_acc:.4f}  macro-F1={rf_f1:.4f}")

In [ ]:
# 3.5 Пробный прогон на 3 чанках (вариант tfmpca).
demo_idx = np.random.RandomState(1).choice(n, size=min(3, n), replace=False)
for i in demo_idx:
    payload = {"states": [{"id": f"chunk_{int(i)}", "state": state_tfmpca[i], "questions": QUESTIONS_NJ}]}
    ans = nj.predict(payload)["states"][0]["answers"]["pattern"]
    pred, conf = ans["choice"], float(ans["probabilities"][ans["choice"]])
    gt = test_labels[int(i)]
    ok = "OK" if pred == gt else "X"
    print(f"[{ok}] true={gt:<12s} pred={pred:<12s} conf={conf:.3f}")

In [ ]:
# 3.6 Zero-shot NanoJev: прогон двух вариантов state по тесту (пакетно).
#     На T4 (Colab) bf16-autocast внутри nj.predict исполняется медленно -> временно
#     форсируем fp16 через nj_predict_fp16 (torch.autocast возвращаем в исходное).
MAX_N = 0            # 0 = вся тестовая выборка; для скорости на T4 можно поставить 300
BATCH = 16           # сколько state-вопросов одним forward (память на T4 хватает)

def nj_predict_fp16(payload, *a, **k):
    _orig = torch.autocast
    def _fp16_acast(device, dtype=torch.bfloat16, *aa, **kk):
        return _orig(device, dtype=torch.float16, *aa, **kk)
    torch.autocast = _fp16_acast
    try:
        return nj.predict(payload, *a, **k)
    finally:
        torch.autocast = _orig

def nanojev_zero_shot(state_rows, n_max, batch=BATCH):
    n_use = len(state_rows) if n_max == 0 else min(n_max, len(state_rows))
    preds, confs = [], []
    for start in tqdm(range(0, n_use, batch), desc="NanoJev zs", unit="batch"):
        idx = list(range(start, min(start + batch, n_use)))
        payload = {"states": [{"id": f"chunk_{int(i)}", "state": state_rows[i],
                               "questions": QUESTIONS_NJ} for i in idx]}
        res = nj_predict_fp16(payload)
        for i, st in zip(idx, res["states"]):
            ans = st["answers"]["pattern"]
            preds.append(ans["choice"])
            confs.append(float(ans["probabilities"][ans["choice"]]))
    return preds, confs

results_zs = {}
for name, rows in [("stats", state_stats), ("tfmpca", state_tfmpca)]:
    p, c = nanojev_zero_shot(rows, MAX_N)
    y_sub = le.inverse_transform(y_test[:len(p)])
    results_zs[name] = {"preds": p, "confs": c,
                        "acc": accuracy_score(y_sub, p),
                        "f1": f1_score(y_sub, p, average="macro")}
    print(f"[NanoJev zero-shot {name}] acc={results_zs[name]['acc']:.4f} "
          f"macro-F1={results_zs[name]['f1']:.4f}  mean_conf={np.mean(c):.4f}")

In [ ]:
# 3.7 Отчёт по варианту tfmpca (матрица ошибок + классификационный отчёт).
y_sub = le.inverse_transform(y_test[:len(results_zs['tfmpca']['preds'])])
print(classification_report(y_sub, results_zs['tfmpca']['preds']))
cm = confusion_matrix(y_sub, results_zs['tfmpca']['preds'])
plt.figure(figsize=(max(6, len(classes)), max(6, len(classes))))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Предсказано"); plt.ylabel("Истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("NanoJev zero-shot (tfmpca state): confusion matrix")
plt.show()

## 4. Франкенштейн: классификационная голова на выходе NanoJev

Голова ставится не на текст, а на **внутреннее представление** NanoJev: для каждого окна строим тот же choice-вопрос, прогоняем candidate-пути через замороженный backbone, берём скрытый вектор **последней позиции** каждого пути (именно он в оригинальной decision-голове превращается в скаляр) и делаем **mean-pool по 4 кандидатам** → вектор `H` (hidden_size Qwen3-0.6B). Поверх него — обучаемая голова `Linear(H, n_classes)` + CE.

**Два варианта:** `stats`-state и `tfmpca`-state — что выигрывает при обученной голове.

In [ ]:
# 4.1 Извлечение фичей с выхода NanoJev (mean-pool векторов последних позиций candidate-путей).
#     Тот же шифр, что в DecisionModel.forward: leaves = hidden[arange, lengths-1];
#     здесь дополнительно mean-pool по кандидатам одного вопроса -> (n_windows, hidden).
#
#     Скорость/удобство:
#       - прогресс по батчам (tqdm);
#       - на T4 (случай Colab) bf16-автокаст работает медленно -> используем fp16,
#         bf16 оставляем только на Hopper+ (compute capability >= 9);
#       - FEAT_BATCH можно поднять/опустить под память GPU.
from predict_toy_decisions import prepare_examples

FEAT_BATCH = 32          # state на один forward (память T4/16GB хватает; при OOM снижайте)

def nj_cast_dtype():
    '''fp16 быстрее на T4/V100, bf16 предпочитаем на A100/H100/H200.'''
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 9 else torch.float16

def nanojev_leaf_features(engine, state_rows, questions, batch=FEAT_BATCH, desc="features"):
    model, tok = engine.model, engine.tokenizer
    cast_dtype = nj_cast_dtype()
    feats = []
    it = tqdm(range(0, len(state_rows), batch), desc=desc, leave=False)
    for start in it:
        payload = {"states": [
            {"id": f"w{start + k}", "state": state_rows[start + k], "questions": questions}
            for k in range(min(batch, len(state_rows) - start))]}
        examples = prepare_examples(payload, tok, engine.limit)
        paths = [ids for ex in examples for ids in ex["leaf_tokens"]]
        if not paths:
            continue
        lengths = torch.tensor([len(ids) for ids in paths], device=engine.device)
        width = int(lengths.max())
        tokens = torch.full((len(paths), width), tok.pad_token_id, dtype=torch.long, device=engine.device)
        for r, ids in enumerate(paths):
            tokens[r, :len(ids)] = torch.tensor(ids, device=engine.device)
        attn = torch.arange(width, device=engine.device)[None, :] < lengths[:, None]
        with torch.inference_mode(), torch.autocast("cuda", dtype=cast_dtype):
            hidden = model.backbone(input_ids=tokens, attention_mask=attn,
                                    use_cache=False).last_hidden_state
            leaves = hidden[torch.arange(len(paths), device=engine.device), lengths - 1]
        leaves = leaves.float()
        off = 0
        for ex in examples:
            m = len(ex["leaf_tokens"])
            feats.append(leaves[off:off + m].mean(0).cpu().numpy())
            off += m
    return np.vstack(feats)

HIDDEN_NJ = int(nj.model.backbone.config.hidden_size)
print("hidden_size NanoJev backbone:", HIDDEN_NJ, "| n_classes:", len(classes))
print("FEAT_BATCH:", FEAT_BATCH, "| autocast dtype:", nj_cast_dtype())

In [ ]:
# 4.2 Фичи выхода NanoJev для train/test в двух вариантах state (Backbone заморожен).
#     Результаты кэшируются в cache_features/*.npz: при повторном запуске (после обучения
#     головы и т.п.) дорогих извлечений не будет. FORCE_RECOMPUTE=True — пересчитать.
import time
FORCE_RECOMPUTE = False
_CACHE_DIR = os.path.join(REPO_DIR, "cache_features")
os.makedirs(_CACHE_DIR, exist_ok=True)

def load_or_compute(name, fn):
    path = os.path.join(_CACHE_DIR, f"feats_{name}.npz")
    if os.path.exists(path) and not FORCE_RECOMPUTE:
        print(f"cache: {name} загружен ({os.path.getsize(path)//1024} KB)")
        return np.load(path)["arr_0"]
    t0 = time.perf_counter()
    arr = fn()
    np.savez_compressed(path, arr)
    print(f"cache: {name} сохранён в {path} ({time.perf_counter() - t0:.0f} c)")
    return arr

state_train_stats  = [chunk_to_state(X_train[i], i) for i in range(len(X_train))]
state_train_tfmpca = [chunk_to_state(X_train[i], i, tfmpca_to_text(Ctr[i])) for i in range(len(X_train))]
print("state lists готовы:", len(state_train_stats), "train /", len(state_stats), "test")

Ftr_stats = load_or_compute("Ftr_stats",
    lambda: nanojev_leaf_features(nj, state_train_stats, QUESTIONS_NJ, desc="feat: train/stats"))
Fte_stats = load_or_compute("Fte_stats",
    lambda: nanojev_leaf_features(nj, state_stats, QUESTIONS_NJ, desc="feat: test/stats"))
Ftr_tf = load_or_compute("Ftr_tf",
    lambda: nanojev_leaf_features(nj, state_train_tfmpca, QUESTIONS_NJ, desc="feat: train/tfmpca"))
Fte_tf = load_or_compute("Fte_tf",
    lambda: nanojev_leaf_features(nj, state_tfmpca, QUESTIONS_NJ, desc="feat: test/tfmpca"))

print("train/test stats  :", Ftr_stats.shape, Fte_stats.shape)
print("train/test tfmpca :", Ftr_tf.shape, Fte_tf.shape)

In [ ]:
# 4.3 Обучение головы Linear(H -> n_classes) на фичах выхода NanoJev.
#     Backbone заморожен; валид-сплит берём из train; стандартизация — только по train.
from torch.utils.data import TensorDataset
import torch.nn.functional as F

def train_head(Ftr, ytr, Fte, yte, tag):
    Xv_tr, Xv_va, yv_tr, yv_va = train_test_split(
        Ftr, ytr, test_size=0.15, random_state=42, stratify=ytr)
    sc = StandardScaler().fit(Xv_tr)
    tr_s, va_s, te_s = sc.transform(Xv_tr), sc.transform(Xv_va), sc.transform(Fte)

    head = nn.Sequential(nn.Dropout(0.3), nn.Linear(Ftr.shape[1], len(classes))).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=3e-4, weight_decay=1e-3)
    bs = 32
    dl_tr = DataLoader(TensorDataset(torch.tensor(tr_s, dtype=torch.float32),
                                     torch.tensor(yv_tr)), batch_size=bs, shuffle=True)
    dl_va = DataLoader(TensorDataset(torch.tensor(va_s, dtype=torch.float32),
                                     torch.tensor(yv_va)), batch_size=bs)

    best_va, best_sd, bad = 0.0, None, 0
    for ep in range(1, 31):
        head.train()
        tot_l, tot_c, tot_n = 0.0, 0, 0
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = head(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            opt.step()
            tot_l += loss.item() * len(yb)
            tot_c += (logits.argmax(-1) == yb).sum().item()
            tot_n += len(yb)
        head.eval()
        va_p = []
        with torch.no_grad():
            for xb, _ in dl_va:
                va_p += head(xb.to(DEVICE)).argmax(-1).tolist()
        va_acc = accuracy_score(yv_va, va_p)
        if ep == 1 or ep % 5 == 0:
            print(f"[{tag}] epoch {ep:2d}: loss={tot_l/tot_n:.4f} "
                  f"train_acc={tot_c/tot_n:.4f} val_acc={va_acc:.4f}")
        if va_acc > best_va:
            best_va, bad = va_acc, 0
            best_sd = {k: v.clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= 6:
                break
    if best_sd is not None:
        head.load_state_dict(best_sd)

    head.eval()
    te_p = []
    with torch.no_grad():
        for xb in DataLoader(torch.tensor(te_s, dtype=torch.float32), batch_size=bs):
            te_p += head(xb.to(DEVICE)).argmax(-1).tolist()
    acc = accuracy_score(yte, te_p)
    f1 = f1_score(yte, te_p, average="macro")
    print(f"[{tag}] TEST acc={acc:.4f} macro-F1={f1:.4f} (best_val={best_va:.4f})")
    return {"acc": acc, "f1": f1, "preds": te_p}

head_stats = train_head(Ftr_stats, y_train, Fte_stats, y_test, "head/stats")
head_tf    = train_head(Ftr_tf,    y_train, Fte_tf,    y_test, "head/tfmpca")

## 5. Итоговая таблица и выводы

In [ ]:
# 5.1 Сводная таблица всех подходов (один и тот же тест из 1.4).
rows = [
    ("RF stats-признаки (бейзлайн)", rf_acc if "rf_acc" in dir() else None,
     rf_f1 if "rf_f1" in dir() else None),
    ("RF / LogReg на TimesFM (потолок)", probe["RandomForest"]["acc"], probe["RandomForest"]["f1"]),
    ("LogReg на TimesFM (потолок)", probe["LogisticRegression"]["acc"], probe["LogisticRegression"]["f1"]),
    ("NanoJev zero-shot: stats state", results_zs["stats"]["acc"], results_zs["stats"]["f1"]),
    ("NanoJev zero-shot: TF-PCA state", results_zs["tfmpca"]["acc"], results_zs["tfmpca"]["f1"]),
    ("NanoJev + голова (stats state)", head_stats["acc"], head_stats["f1"]),
    ("NanoJev + голова (TF-PCA state)", head_tf["acc"], head_tf["f1"]),
]

print("=" * 62)
print(f"{'Подход':<44}{'acc':>8}{'F1_macro':>10}")
print("-" * 62)
for name, a, f in rows:
    if a is None or f is None:
        print(f"{name:<44}{'—':>8}{'—':>10}")
    else:
        print(f"{name:<44}{a:>8.4f}{f:>10.4f}")
print("=" * 62)

In [ ]:
# 5.2 Сводный бар-чарт по accuracy.
import numpy as np
import matplotlib.pyplot as plt

plot_items = [(r[0], r[1]) for r in rows if r[1] is not None]
labels = [p[0] for p in plot_items]
vals = [p[1] for p in plot_items]

plt.figure(figsize=(11, max(5, 0.5 * len(labels))))
bars = plt.barh(labels, vals, color="#4C72B0")
plt.xlabel("accuracy (test)")
plt.title("TimesFM → NanoJev: сравнение подходов")
plt.xlim(0, 1.0)
for b, v in zip(bars, vals):
    plt.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

### Что здесь вообще происходит — коротко

- **RF stats** — «честный» бейзлайн на тех же 11 статистиках, что идут в state.
- **Потолок на TimesFM** — RF/LogReg на сырых 1280-мерных эмбеддингах (после StandardScaler). Это максимум, чего признаки TimesFM способны достичь без Jev-механик.
- **NanoJev zero-shot** — «передали признаки в state модели», но модель **не обучалась** на этих классах; чистый трансфер чекпойнта с игровой предметной области. Скорее всего заметно хуже потолка.
- **NanoJev + голова** — то же самое, но теперь голова **обучена** на train-сплите поверх замороженного вектора `H`. Ожидаемый результат: сильно лучше zero-shot, но всё ещё ниже RF на эмбеддинг, если причин посадить модель на сырые эмбеддинги (голова видит только ~представление текста, в которое признаки попали через PCA и троттлинг текста).

### Важные оговорки

1. **Транспорт признаков через текст (tfmpca) — главный узкое горлышко.** 16 компонент PCA, округлённых до 3 знаков и прогнанных через BPE-токенизатор, несут лишь часть информации. Это честный «text bottleneck» — так же, как Jev работает вообще без генерации текста, здесь признаки упакованы в текст.
2. **Нет утечки:** PCA и StandardScaler калибруются только на train (ячейки 2.3–2.4).
3. **Если вы хотите максимального качества** — RF/LogReg на эмбеддинг TimesFM (потолок) делает это напрямую и без текста. NanoJev-механика здесь добавляет value only if вы хотите calibrated-вероятности поверх одного и того же state (что и есть суть Jev/System One: распределение, а не строка-лейбл).
4. Обучение головы простое (Linear + CE, early-stop по val-accuracy, 6 без улучшений — стоп). Варианты для улучшения: полный fine-tune backbone при низком LR, добавление в state raw-признаков помимо PCA, multi-task (класс + паттерн-описание).